# Boat Brain Support


This file supports Boat Brain, a small Arduino based (ESP32) computer that can *do everything* on the boat.

- Connect on NMEA2k and log all the sailing instrument data to SD card.
- Provide a high frequency WIFI gateway to all the critical instruments.
- Retransmit NMEA2k messages that are processed.
- Collect logs and upload on request.

## Boat Brain

- Very fast interface to all boat instruments that can communicate with an an onboard
  iphone.
  - Data is sent out by WIFI and/or Bluetooth.
  - Device sits on the N2k bus and consumes all messages.
  - Device captures logs and stores them for later download.
  - Device can upload the logs over wifi.

- A new set of virtual instruments that provide filtered and improved boat data, and that can be
  displayed on the Triton and Zeus.
  - Real-time processing of messages and resending filtered versions.
  - First candidate is TWD, TWS.
  - Better port/stbd calibrated boat speed.
  - Upwind vs. downwind AWS (and therefore TWS).
  - Dynamic damping (which is how I do most filtering)
    - If the residual is too high then you decrease the damping to catch up.
    - Two parameters.  Damping and the threshold.

This all replaces the RPi, and it is lower power and can be left in the boat!!

## Testing Design

### Data Flow

In order to support testing of the software and the device off the boat.

- Create a hardware test bed. Two ESP32s, connected by CAN, both running N2k software.
  - Boat Brain (BB) which does all the above tasks
  - ActisenseRelay (AR), which reads N2k messages from the serial bus and resends them on CAN
- The BB computer believes it is on the boat, receiving messages from AR in real time.
- Two pieces of Python code.
  - One that reads an actisense log (from the Raspberry Pi) and then send it to AR.
  - The second which connects to BB to get BoatState messages.
- We can then compare the output of BB with the logs sent to AR.
- This also tests BB logging to the SD card.

One neat example was measuring end to end latency.  The AR python code stamped the real actual time on top of the System Time message.  This was sent to AR, which is sends this on to BB, which sets the BB real time clock.  Times are then send back to the BB Python code.  The latency can then be measured by comparing the timestamps.  

Using this technique we can see that over 50 messages we see time interval between messages is 100 msecs with just a few outliers.  The end to end latency is just over half of the message interval, which is close to the optimal

- Message times    avg =   0.100, min =   0.067, max =   0.133
- Time latency     avg =   0.067, min =   0.040, max =   0.104

These number were measured when the BB is acting as an access point (its own Wifi network).  This is **not** the numbers you get when both machines are connected to shared access point (in my case some Eero mesh devices).  In that case there is much more buffering, with long gaps between messages.

### Functionality




In [ ]:
import numpy as np

# These are libraries written for RaceAnalysis
from Libraries.global_variables import G
from Libraries.nbutils import display_markdown, display
import Libraries.race_logs as race_logs
import Libraries.metadata as m
import Libraries.process as p
import Libraries.analysis as a
import Libraries.chart as ch
import Libraries.utils as utils
import Libraries.quick_plot as qp

# import Libraries.nbutils

# Initialize for Seattle.
G.init_seattle(logging_level="INFO")

import itertools as it
import datetime
import pandas as pd

import Libraries.boat_brain as bb
import Libraries.canboat as cb

import matplotlib.pyplot as plt

import estimator as est


In [ ]:
# 

# create a two way serial interface where we can send lines and receive lines
lines = [
    "2023-10-29T18:53:08.496Z,6,129539,127,255,8,00,d3,32,00,50,00,ff,7f",
    "2023-10-29T18:53:08.553Z,7,127258,127,255,8,00,f6,ff,ff,e7,09,ff,ff",
    "2023-10-29T18:53:08.553Z,3,127257,128,255,8,00,ff,7f,77,04,5d,08,ff",
    "2023-10-29T18:53:08.553Z,2,127250,4,255,8,ff,dc,0b,ff,7f,ff,7f,fd",
    "2023-10-29T18:53:08.553Z,2,65280,16,255,8,13,99,04,05,00,00,02,00",
    "2023-10-29T18:53:08.553Z,6,65420,3,255,8,41,9f,10,8d,57,0b,f1,ff",
    "2023-10-29T18:53:08.553Z,2,127245,15,255,8,ff,ff,ff,7f,28,05,ff,ff",
    "2023-10-29T18:53:08.553Z,2,127251,4,255,8,ff,14,aa,05,00,ff,ff,ff",
]


#example system time lines
system_time_lines = [
"2023-10-29T16:13:57.631Z,3,126992,5,255,8,ba,f0,cb,4c,10,d8,cc,22",
"2023-10-29T16:13:58.295Z,3,126992,127,255,8,00,f0,cb,4c,a8,12,cd,22",
"2023-10-29T16:13:58.628Z,3,126992,5,255,8,bb,f0,cb,4c,20,ff,cc,22",
"2023-10-29T16:13:59.224Z,3,126992,127,255,8,00,f0,cb,4c,b8,39,cd,22",
"2023-10-29T16:13:59.617Z,3,126992,5,255,8,bc,f0,cb,4c,30,26,cd,22",
"2023-10-29T16:14:00.246Z,3,126992,127,255,8,00,f0,cb,4c,c8,60,cd,22",
"2023-10-29T16:14:00.617Z,3,126992,5,255,8,bd,f0,cb,4c,40,4d,cd,22",
"2023-10-29T16:14:01.289Z,3,126992,127,255,8,00,f0,cb,4c,d8,87,cd,22",
"2023-10-29T16:14:01.625Z,3,126992,5,255,8,be,f0,cb,4c,50,74,cd,22",
]


In [ ]:
# convert unix time back to datetime
utime = 1698596065.00
utime = 1698596104.00
utime = 1704508597.45
utime = 1704509092.471
utime = 1704534025.20

print(datetime.datetime.fromtimestamp(utime))

In [ ]:
import importlib
importlib.reload(bb)


In [ ]:
nlf = sorted(p.named_log_files())
nlf[-10:]

big_logs = [
"actisense_2023_08_06_10_04_46_6141.log.gz",
"actisense_2023_08_06_17_28_49_1458.log.gz",
"actisense_2023_08_14_20_54_29_20035.log.gz",
]

In [ ]:
if False:
    # converts some compressed logs to json
    base_dir = "/Users/viola/Python/sailing/Data/CompressedLogs/"

    res = []
    for log in big_logs:
        print(log)
        json_path = p.json_from_named_file(base_dir + log)
        res.append(json_path)

    res

In [ ]:
json_files = [   '/Users/viola/Python/sailing/Data/CompressedLogs/2023-10-29_09:13.json',
    '/Users/viola/Python/sailing/Data/CompressedLogs/actisense_2023_08_06_10_04_46_6141.json',
 '/Users/viola/Python/sailing/Data/CompressedLogs/actisense_2023_08_06_17_28_49_1458.json',
 '/Users/viola/Python/sailing/Data/CompressedLogs/actisense_2023_08_14_20_54_29_20035.json']

In [ ]:
pgns_to_keep = {
   130306: "Wind Data",
            126992: 'System Time',
            127245: 'Rudder',
            127250: 'Vessel Heading',
            127251: 'Rate of Turn',
            127257: 'Attitude (pitch, roll)',
            127258: 'Magnetic Variation',
            128259: 'Speed',
            129025: 'Position, Rapid Update',
            129026: 'COG & SOG, Rapid Update',
            130306: 'Wind Data'
        }

In [ ]:
date = '2023-10-29'
df, race = race_logs.read_date(date)


In [ ]:
# read and process the json file to create our normal RegattaAnalysis data frame

# bdf = cb.json_to_data_frame(json_file)
# pdf = race_logs.process_pickled_data(bdf)

pdf = df

In [ ]:
pdf.columns

In [ ]:
import importlib
importlib.reload(est)

In [ ]:
# code is a bit slow, so we will slice the data frame to a smaller range
epdf = pdf.copy().iloc[150000:230000]

def na(val):
    return np.asarray(val)

res = {}

# for a in [0.8, 0.85, 0.9]:
#     for m in [1.0, 1.05, 1.1]:
for a in [0.8, 0.85, 0.9]:
    for m in [1.0, 1.05, 1.1]:
        key = f"{a:.2f}, {m:.2f}"
        print(key)
        mepdf = epdf.copy()
        mepdf.rawa = np.where(abs(mepdf.awa) < 90, mepdf.rawa * a, mepdf.rawa)
        mepdf.raws = np.where(abs(mepdf.awa) < 90, mepdf.raws * m, mepdf.raws)
        est.estimate_true_wind(mepdf, eps_tws=0.02, eps_twd=0.005, time_delay=12, eps=0.01)
        res[key] = (na(mepdf.ptws), na(mepdf.ptwd))
# mepdf.raws = np.where(abs(mepdf.awa) > 90, mepdf.raws * 0.9, mepdf.raws)

# est.estimate_true_wind(mepdf, eps_tws=0.02, eps_twd=0.08, time_delay=12)


In [ ]:
def sa(d):
    return p.sign_angle(d)

def rad(d):
    return d * np.pi / 180

def deg(d):
    return d * 180 / np.pi

nres = {}
nres["0.85, 1.05"] = res["0.85, 1.05"]
res = nres

data_to_plot = [sa(res[k][1]) for k in res.keys()]
names = [k for k in res.keys()]

#  qp.plot(None, data_to_plot+[mepdf.awa, sa(mepdf.twd), mepdf.awsres, deg(mepdf.awares)], names + "awa, twd, awsres, awares".split(", "))
# qp.plot(None, data_to_plot+[mepdf.awa, sa(mepdf.twd)], names + "awa, twd, awsres, awares".split(", "))

qp.plot(None, data_to_plot+[mepdf.awa, sa(mepdf.twd), sa(mepdf.hdg)], names + "awa, twd, hdg".split(", "))
# qp.plot(None, data_to_plot+[mepdf.awa, sa(mepdf.twd)], names + "awa, twd, awsres, awares".split(", "))


In [ ]:
mepdf.awares.min() , mepdf.awares.max()

In [ ]:
est.torad(75)

In [ ]:
epdf['ptws'] = mepdf.ptws
epdf['ptwd'] = mepdf.ptwd
epdf['awsres'] = mepdf.awsres
epdf['awares'] = mepdf.awares

In [ ]:
def sa(d):
    return p.sign_angle(d)

# Plot two figures on the same plot, linking the x axis
fig = plt.figure()
pltdf = epdf
# two axes on the same figure
ax1, ax2 = fig.subplots(2, 1, sharex=True)
qp.quick_plot_ax(ax1, pltdf.index, (pltdf.tws, pltdf.ptws, pltdf.aws, pltdf.sog, pltdf.awsres), "tws, ptws, aws, sog, awsres")
qp.quick_plot_ax(ax2, pltdf.index, (sa(pltdf.twd), sa(pltdf.ptwd), pltdf.awa, sa(pltdf.cog), deg(pltdf.awares), sa(pltdf.boat_twd)), "twd, ptwd, awa, cog, awares, boat")
# qp.quick_plot_ax(ax2, pltdf.index, (pltdf.awa, pltdf.twd, pltdf.ptwd, pltdf.cog, pltdf.awares), "awa, twd, ptwd, cog")

In [ ]:
plt.close('all')

In [ ]:
qp.plot(pltdf.index, (sa(pltdf.twd), sa(pltdf.ptwd), pltdf.awa, sa(pltdf.boat_twd)), "twd, ptwd, awa, boat")

In [ ]:
import sympy.utilities.codegen
from estimator import *

from sympy.printing.cxx import tocode

# display(tocode(awa))
# display(cxxcode(aws))

awa_npfunc = lambdify((TWS, TWD, HDG, SOG, COG), awa, 'numpy')
aws_npfunc = lambdify((TWS, TWD, HDG, SOG, COG), aws, 'numpy')

In [ ]:

def awa_residuals(df):
    sdf = df.iloc[5000:]
    awa1 = sa(todeg(awa_npfunc(sdf.tws, torad(sdf.twd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))))
    awa2 = sa(todeg(awa_npfunc(sdf.ptws, torad(sdf.ptwd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))))
    awa3 = sa(todeg(awa_npfunc(sdf.boat_tws, torad(sdf.boat_twd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))))
    # sum of squares of residuals
    print("AWA sum of squares of residuals")
    print(f"grad:  {np.sum((awa1 - sdf.rawa)**2)/len(sdf.awa)}")
    print(f"gauss: {np.sum((awa2 - sdf.rawa)**2)/len(sdf.awa)}")
    print(f"boat:  {np.sum((awa3 - sdf.rawa)**2)/len(sdf.awa)}")
    qp.plot(None, (sdf.awa, awa1, awa2, awa3), "awa grad gauss boat".split())
 
def aws_residuals(df):
    sdf = df.iloc[5000:]
    aws1 = aws_npfunc(sdf.tws, torad(sdf.twd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))
    aws2 = aws_npfunc(sdf.ptws, torad(sdf.ptwd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))
    aws3 = aws_npfunc(sdf.boat_tws, torad(sdf.boat_twd), torad(sdf.hdg+15.2), sdf.sog, torad(sdf.cog))
    # sum of squares of residuals
    print("AWS: sum of squares of residuals")
    print(f"grad:  {np.sum((aws1 - sdf.raws)**2)/len(sdf.awa)}")
    print(f"gauss: {np.sum((aws2 - sdf.raws)**2)/len(sdf.awa)}")
    print(f"boat:  {np.sum((aws3 - sdf.raws)**2)/len(sdf.awa)}")

awa_residuals(epdf)
aws_residuals(epdf)   


In [ ]:
case = dict(TWD = 0,
    TWS = 10,
    HDG = 45,
    COG = 45,
    SOG = 8)

def case_rad(c):
    res = {}
    for k in c.keys():
        if k in c and k in ['TWD', 'HDG', 'COG', 'AWA']:
            res[k] = torad(c[k])
        else:
            res[k] = c[k]
    return res

case_rad(case)

In [ ]:

awa, aws = awa_func(**case_rad(case)), aws_func(**case_rad(case))
print(awa, aws)


ncase = case.copy()
ncase['AWA'] = todeg(awa)
ncase['AWS'] = aws

print(ncase)


In [ ]:
import

In [ ]:
importlib.reload(est)


In [ ]:
ncase['TWD'] = 10
ncase['TWS'] = 20
ncase['AWS'] = aws * 1.0
ncase['AWA'] = todeg(awa) * 1.1
print(case)
print(ncase)

ntwd, ntws = est.gauss_newton(**ncase)
print(sa(ntwd), ntws)

ncase['AWS'] = aws * 1.0
ncase['AWA'] = todeg(awa) * 0.9
print(case)
print(ncase)

ntwd, ntws = est.gauss_newton(**ncase)
print(sa(ntwd), ntws)

In [ ]:
tcase = ncase.copy()
tcase['TWD'] = ntwd
tcase['TWS'] = ntws

del tcase['AWA']
del tcase['AWS']

print(tcase)
print(awa_func(**tcase), aws_func(**tcase))

In [ ]:



qp.plot(None, (epdf.aws, aws1, aws2, aws3), "aws grad gauss boat".split())

In [ ]:
qp.plot(None, (epdf.awa, ttt1, ttt2, ttt3), "awa grad gauss boat".split())

In [ ]:
import arrow
skip  = 200000
count = 10000
if True:
    pgn_whitelist = set(pgns_to_keep.keys())
    sliced = it.islice(cb.file_lines(json_file), skip, skip+count)
    records = cb.json_records(sliced, cb.pgn_filter(pgn_whitelist), 1)
    # Throw out the first trim records, just to make sure things are working

    rows = []
    row = {}
    for record in records:
        tr = cb.transform_record(record)
        fields = tr['fields']
        if 'reference' in fields and fields['reference'] == 'Apparent':
            row['timestamp'] = record['timestamp']
            row['aws'] = fields['wind_speed']
            row['awa'] = fields['wind_angle']
            rows += [row]
    df = pd.DataFrame(rows)

df.awa = p.sign_angle(df.awa)

df


In [ ]:
# show the types of the df
df.dtypes
# convert the timestamp to a datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

In [ ]:
# compute the delta between each timestamp, in seconds
df['delta'] = df['timestamp'].diff()
# convert to seconds
df['delta'] = df['delta'].dt.total_seconds()
print(f"mean: {df['delta'].mean()}, min: {df['delta'].min()}, max: {df['delta'].max()}")


In [ ]:
plt.figure()
plt.hist(df['delta'], bins=100)
plt.show()

In [ ]:
import Libraries.quick_plot as qp

qp.plot(df.index, (df.awa, ))


In [ ]:


# func to smooth the wind angle, it has two parts: exponential smoothing, with an adjustable gain that reduces smoothing if the residual is too large.
def smooth_angles(data, base_gain=0.1, max_residual=10):
    smoothed = np.zeros(len(data))
    smoothed[0] = data[0]
    gain = base_gain
    for index in range(1, len(data)):
        val = data[index]
        residual = p.angle_diff_d(val, smoothed[index-1])
        if abs(residual) > max_residual:
            gain += base_gain
        else:
            # decay back to base_gain
            gain = 0.7 * gain + 0.3 * base_gain
        smoothed[index] = p.sign_angle(smoothed[index-1] + gain * residual)
    return smoothed



# func to smooth the wind angle, it has two parts: exponential smoothing, with an adjustable gain that reduces smoothing if the residual is too large.
def smooth_data(data, base_gain=0.1, max_residual=10):
    smoothed = np.zeros(len(data))
    smoothed[0] = data[0]
    gain = base_gain
    for index in range(1, len(data)):
        val = data[index]
        residual = val - smoothed[index-1]
        if abs(residual) > max_residual:
            gain += base_gain
        else:
            # decay back to base_gain
            gain = 0.7 * gain + 0.3 * base_gain
        smoothed[index] = smoothed[index-1] + gain * residual
    return smoothed


In [ ]:
df['sawa'] = smooth_angles(df['awa'].values, base_gain=0.1, max_residual=10)

df['saws'] = smooth_data(df['aws'].values, base_gain=0.01, max_residual=0.8)

In [ ]:
# qp.plot(df.index, (df.awa, df.sawa))
qp.plot(df.index, (df.aws, df.saws))

In [ ]:
# read a gzip pickle file
dir = "/Users/viola/tmp/"
file = dir + '2024_01_08_14_58_52.pd.gz'
import gzip
import pickle
with gzip.open(file, 'rb') as f:
    data = pickle.load(f)
rtdf = pd.DataFrame(data)

In [ ]:
# stats for the columns
rtdf.describe()

In [ ]:
import Libraries.signals as sig

In [ ]:
qp.plot(rtdf.index, (rtdf.twd, sig.sign_angle(rtdf.ptwd)), "twd, ptwd")

In [ ]:
# Plot two figures on the same plot, linking the x axis
fig = plt.figure()
# two axes on the same figure
ax1, ax2 = fig.subplots(2, 1, sharex=True)
# qp.quick_plot_ax(ax1, rtdf.index, (sig.sign_angle(rtdf.twd), rtdf.ptwd, rtdf.saws), "twd, ptwd, saws")
qp.quick_plot_ax(ax1, rtdf.index, (rtdf.tws, rtdf.ptws, rtdf.saws), "twd, ptwd, saws")
qp.quick_plot_ax(ax2, rtdf.index, (rtdf.awa, rtdf.sawa), "awa, sawa")

In [ ]:
qp.plot(rtdf.index, (rtdf.awa, rtdf.sawa))

In [ ]:
rtdf['psawa'] = smooth_angles(rtdf['awa'].values, base_gain=0.1, max_residual=10)

In [ ]:
qp.plot(rtdf.index, (rtdf.awa, rtdf.sawa, rtdf.psawa), "awa sawa psawa".split())

In [ ]:
import sympy.utilities.codegen
from estimator import *

import sympy as sp
from sympy.printing.cxx import cxxcode
from sympy.printing.python import python

tocode = python

print(python(awa))
print(python(aws))
print(python(d_aws_d_twd))
print(python(d_awa_d_twd))
print(python(d_aws_d_tws))
print(python(d_awa_d_tws))


from sympy.printing.numpy import NumPyPrinter as npcode

awa_npfunc = lambdify((TWS, TWD, HDG, SOG, COG), awa, 'numpy')


def aws_hc(TWS, TWD, HDG, SOG, COG):
   return -HDG + np.atan2(SOG*np.sin(COG) + TWS*np.sin(TWD), SOG*np.cos(COG) + TWS*np.cos(TWD))

def awa_hc(TWS, TWD, HDG, SOG, COG):
   return np.sqrt((-SOG*np.sin(COG) - TWS*np.sin(TWD))**2 + (-SOG*np.cos(COG) - TWS*np.cos(TWD))**2)

def daws_dtwd_hc(TWS, TWD, HDG, SOG, COG):
    return SOG*TWS*np.sin(COG - TWD)/np.sqrt(SOG**2 + 2*SOG*TWS*np.cos(COG - TWD) + TWS**2)
# e = (SOG*np.cos(COG - TWD) + TWS)/np.sqrt(SOG**2 + 2*SOG*TWS*np.cos(COG - TWD) + TWS**2)
# e = -SOG*np.sin(COG - TWD)/(SOG**2 + 2*SOG*TWS*np.cos(COG - TWD) + TWS**2)
def dawa_dtwd_hc(TWS, TWD, HDG, SOG, COG):
    return TWS*(SOG*np.cos(COG - TWD) + TWS)/(SOG**2 + 2*SOG*TWS*np.cos(COG - TWD) + TWS**2)

In [ ]:
from sympy.printing.numpy import NumPyPrinter as npcode

awa_npfunc = lambdify((TWS, TWD, HDG, SOG, COG), awa, 'numpy')


In [ ]:
len(np.zeros(10))

In [ ]:
from sympy.utilities.codegen import codegen

# Generate C++ code
[(c_name, c_code), (h_name, c_header)] = codegen(
    [('aws_function', aws), ('awa_function', awa),
     ('d_aws_d_twd_function', d_aws_d_twd), ('d_awa_d_twd_function', d_awa_d_twd),
     ('d_aws_d_tws_function', d_aws_d_tws), ('d_awa_d_tws_function', d_awa_d_tws),
     ],
    # language='C',
    language='Python',
    project='WindCalc'
)

# c_code contains the C++ code
print(c_code)

In [ ]:
from sympy.utilities.codegen import codegen

# Generate C++ code
[(c_name, c_code), (h_name, c_header)] = codegen(
    [('aws_function', aws), ('awa_function', awa),
     ('d_aws_d_twd_function', d_aws_d_twd), ('d_awa_d_twd_function', d_awa_d_twd),
     ('d_aws_d_tws_function', d_aws_d_tws), ('d_awa_d_tws_function', d_awa_d_tws),
     ],
    language='C',
    project='WindCalc'
)

# c_code contains the C++ code
print(c_code)


In [ ]:
[(c_name, c_code), (h_name, c_header)] = est.generate_code()

In [ ]:
print(c_header)

In [ ]:
import estimator
importlib.reload(estimator)
from estimator import *


In [ ]:

def test_functions(f1, f2, count):
    # Test the two functions against each other
    for i in range(count):
        tws = np.random.uniform(0.1, 20)
        twd = np.random.uniform(0, 2 * math.pi)
        hdg = np.random.uniform(0, 2 * math.pi)
        sog = np.random.uniform(0.1, 20)
        cog = np.random.uniform(0, 2 * math.pi)
        v1 = f1(tws, twd, hdg, sog, cog)
        v2 = f2(tws, twd, hdg, sog, cog)
        if abs(v1 - v2) > 0.001:
            print(f"FAIL {v1:.8f} {v2:.8f}")
            print(tws, twd, hdg, sog, cog)
            raise Exception("FAIL")
            

def test_all_functions():
    print("Test awa")
    test_functions(awa_func, awa_hc, 1000)
    print("Test aws")
    test_functions(aws_func, aws_hc, 1000)
    print("Test daws_dtwd")
    test_functions(daws_dtwd_func, daws_dtwd_hc, 1000)
    print("Test dawa_dtwd")
    test_functions(dawa_dtwd_func, dawa_dtwd_hc, 1000)
    print("Test daws_dtws")
    test_functions(daws_dtws_func, daws_dtws_hc, 1000)
    print("Test dawa_dtws")
    test_functions(dawa_dtws_func, dawa_dtws_hc, 1000)


test_all_functions()

In [ ]:
print(python(aws))